<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/13-cleaning.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 13 — Cleaning and Reshaping

Companion to [the chapter](https://www.ai.biz/books/python-primer/cleaning-and-reshaping/).


In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(7)


## 0. A deliberately messy dataset


In [ ]:
n = 800
df = pd.DataFrame({
    'order_id': list(range(n-20)) + list(range(20)),   # 20 duplicated ids
    'city': rng.choice(['Delhi',' delhi','DELHI','Tokyo','tokyo '], n),
    'age': rng.normal(40, 12, n).round(),
    'income': rng.lognormal(10, 0.5, n).round(-2),
    'updated_at': pd.to_datetime('2026-01-01') + pd.to_timedelta(rng.integers(0,60,n), 'D'),
})
df.loc[rng.choice(n, 60, replace=False), 'income'] = np.nan
df.loc[rng.choice(n, 12, replace=False), 'age'] = -1        # encoded missing
df.loc[rng.choice(n, 5, replace=False), 'age'] = 999        # encoded missing
print(df.shape); df.head()


## 1. Profile before you fix anything


In [ ]:
def profile(d):
    print(f'shape: {d.shape}')
    print(f'exact duplicate rows: {d.duplicated().sum()}')
    return pd.DataFrame({
        'dtype': d.dtypes,
        'missing_pct': (d.isna().mean()*100).round(1),
        'unique': d.nunique(),
    }).sort_values('missing_pct', ascending=False)

profile(df)


## 2. Impossible values hide as encoded missing


In [ ]:
print(df.age.describe())
print()
bad = ~df.age.between(0, 120)
print(f'{bad.sum()} impossible ages:', sorted(df.loc[bad,'age'].unique()))
print()
print('mean age BEFORE fixing:', df.age.mean().round(1))
df.loc[bad, 'age'] = np.nan
print('mean age AFTER fixing :', df.age.mean().round(1))


`-1` and `999` were a system's way of writing 'unknown'. Left in, they poison every average.


## 3. Missingness can be the signal


In [ ]:
mask = df.income.isna()
print('are rows with missing income different?')
print(df.groupby(mask)[['age']].mean().round(1))
print()
# Keep the flag, then fill
df['income_was_missing'] = df.income.isna().astype(int)
df['income'] = df.income.fillna(df.income.median())
print('flag kept:', df.income_was_missing.sum(), 'rows')


## 4. Duplicates: look before you drop


In [ ]:
dupes = df[df.duplicated('order_id', keep=False)].sort_values('order_id')
print(f'{df.order_id.duplicated().sum()} duplicate order_ids')
print(dupes[['order_id','city','updated_at']].head(6))
print()
before = len(df)
df = df.sort_values('updated_at').drop_duplicates('order_id', keep='last')
print(f'{before} -> {len(df)} rows, keeping the most recent per order')


`keep='first'`, `keep='last'` and `keep=False` are three different business decisions.


## 5. Outliers: IQR beats z-score on skewed data


In [ ]:
q1, q3 = df.income.quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
iqr_out = ((df.income < lo) | (df.income > hi)).sum()

z = (df.income - df.income.mean()) / df.income.std()
z_out = (z.abs() > 3).sum()

print(f'IQR method flags     : {iqr_out}')
print(f'z-score method flags : {z_out}')
print()
print('The mean and SD are themselves distorted by the outliers,')
print('which is the circularity the quartile method avoids.')


## 6. Reshaping: wide to long and back


In [ ]:
wide = pd.DataFrame({'city':['Delhi','Tokyo'],
                     '2023':[33.8,36.5], '2024':[33.4,35.3], '2025':[33.8,37.1]})
long = wide.melt(id_vars=['city'], var_name='year', value_name='population')
print(long)
print()
print(long.pivot(index='city', columns='year', values='population'))


## 7. A re-runnable cleaning function with checks


In [ ]:
def clean(raw):
    d = raw.copy()
    d['city_key'] = d.city.str.strip().str.lower()
    d.loc[~d.age.between(0,120), 'age'] = np.nan
    return d.sort_values('updated_at').drop_duplicates('order_id', keep='last')

def check(d):
    assert d.order_id.is_unique, 'order_id not unique'
    assert d.age.dropna().between(0,120).all(), 'impossible age'
    print('all checks passed')

c = clean(df); check(c)
print('city categories after normalising:', sorted(c.city_key.unique()))


Idempotent, testable, and next month's surprise becomes a crash rather than a wrong number.


## Try it yourself

1. Run `clean` twice on its own output. The result should be identical.
2. Add a row with `age = 200` and confirm `check` raises.
3. Compare `keep='first'` against `keep='last'` and count how many rows differ.
